# Identity 격리를 적용한 Amazon Bedrock AgentCore Runtime 및 AgentCore Memory Agent

## 개요

이 튜토리얼에서는 AgentCore Runtime과 AgentCore Memory를 사용하여 사용자 격리가 적용된 첫 번째 Memory 지원 Agent를 만드는 방법을 살펴봅니다. 세션 내에서 이전 상호작용을 기억하여 여러 상호작용에 걸쳐 사용자와 더 자연스럽고 컨텍스트에 맞는 대화를 제공하는 간단한 "Hello World" 대화형 Agent를 구축합니다.

Memory는 효과적인 대화형 Agent를 만드는 핵심 구성 요소입니다. Agent가 컨텍스트를 유지하고 사용자 선호도를 기억하며 시간이 지나도 일관된 응답을 제공할 수 있게 해 줍니다. Memory가 없으면 Agent는 상호작용마다 처음부터 시작해야 하므로 사용자 경험이 단절됩니다.

이 구현은 사용자 Identity propagation이 적용된 Amazon Bedrock AgentCore Memory를 활용하여 인증된 사용자 자격 증명에 따라 Memory를 자동 분할하고 개별 사용자마다 안전하고 격리된 Memory 공간을 생성합니다.

### 튜토리얼 세부 정보


| 정보         | 세부 정보                                                          |
|:--------------------|:-----------------------------------------------------------------|
| 튜토리얼 유형       | Hello World / 소개                                       |
| Agent 유형          | 단일 대화형 Agent                                      |
| Agentic Framework   | Strands Agents                                                   |
| LLM 모델           | Anthropic Claude Haiku 3.5                                      |
| 주요 기능        | AgentCore Runtime, Memory 통합                            |
| 예제 난이도  | 중급                                                         |
| 사용 SDK            | boto3, bedrock-agentcore, bedrock-agentcore-starter-toolkit      |

### 학습 내용

이 튜토리얼에서는 다음 내용을 학습합니다.
1. AgentCore Memory를 사용하여 Agent용 Memory 리소스를 생성하는 방법
2. 대화 기록을 저장하고 검색하는 Memory Hook을 구현하는 방법
3. 확장 가능한 프로덕션 사용을 위해 Agent를 AgentCore Runtime에 배포하는 방법
4. 세션 관리 기능으로 Agent를 테스트하고 메모리 지속성을 검증하는 방법
5. 사용자 Identity를 처리하고 서로 다른 사용자 간 메모리 격리를 보장하는 방법


### 아키텍처

이 Hello World 예제는 Memory가 통합되어 AgentCore Runtime에 배포된 간단한 대화형 Agent를 보여 줍니다.

<div style="text-align:left">
    <img src="runtime-memory-identity.png" width="90%"/>
</div>


## 0. 사전 요구 사항

이 튜토리얼을 실행하려면 다음 항목이 필요합니다.
* Python 3.10 이상
* Bedrock, ECR, IAM 및 Cognito에 적절한 권한이 있는 AWS 자격 증명
* Amazon Bedrock 모델 액세스(Claude 3.5 Haiku)
* Amazon Bedrock AgentCore SDK 및 종속성

먼저 필요한 라이브러리를 설치합니다.

In [ ]:
!pip install -qUr requirements.txt

### 환경 설정

필요한 라이브러리를 가져오고 환경을 구성합니다. 다음 항목을 사용합니다.
- AWS 서비스 상호작용을 위한 `boto3`
- Agent Memory 관리를 위한 `bedrock_agentcore.memory`
- 인증 설정을 위한 여러 utility 함수

In [ ]:
# 가져오기
import os
import time
import boto3
import uuid
import logging
from bedrock_agentcore.memory import MemoryClient
from utils import setup_cognito_user_pool

# 구성
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(name)s: %(message)s")
logger = logging.getLogger("runtime-memory-agent")
REGION = os.getenv("AWS_REGION", "us-west-2")
memory_client = MemoryClient(region_name=REGION)

## 1. Amazon Cognito User Pool 생성

이 섹션에서는 Amazon Cognito User Pool과 사용자를 생성합니다. Cognito는 Agent에 사용자 인증 및 Identity 관리를 제공하여 각 사용자의 대화 기록에 해당 사용자만 액세스하도록 보장합니다.

`setup_cognito_user_pool` 함수는 다음을 수행합니다.
1. Cognito User Pool이 없으면 생성
2. 인증용 app client 설정
3. 임시 password를 사용하는 테스트 사용자 2명 생성
4. 테스트용 access token 생성

In [ ]:
print("Setting up Amazon Cognito user pool and users...")
cognito_config = setup_cognito_user_pool(region=REGION)
print("Cognito setup completed ✓")

## 2. Memory 리소스 생성

이 섹션에서는 Agent가 대화 기록을 저장할 Memory 리소스를 생성합니다. Memory를 사용하면 Agent가 과거 상호작용을 기억하고 컨텍스트를 유지하여 시간이 지나도 더 일관된 응답을 제공할 수 있습니다.

이 예제에서는 추가 장기 strategy 없이 간단한 단기 Memory 리소스를 생성합니다. Memory는 모든 대화 메시지를 저장하여 AgentCore Runtime에서 세션이 종료된 후에도 세션을 이어 갈 때 Agent가 이전 상호작용을 기억하도록 합니다.

In [ ]:
from botocore.exceptions import ClientError

# 이 리소스의 고유 식별자 생성
unique_id = str(uuid.uuid4())[:8]
memory_name = f"RuntimeIdentityMemoryAgent_{unique_id}"

try:
    # strategy 없이 Memory 리소스 생성(단기 메모리만 사용)
    memory = memory_client.create_memory_and_wait(
        name=memory_name,
        strategies=[],  # 단기 메모리에는 strategy를 사용하지 않음
        description="Short-term memory for AgentCore Runtime agent authenticated with AgentCore Identity",
        event_expiry_days=7,  # 단기 메모리 보존 기간
    )
    memory_id = memory["id"]
    logger.info(f"✅ Created memory: {memory_id}")
except ClientError as e:
    logger.info(f"❌ ERROR: {e}")
    if e.response["Error"]["Code"] == "ValidationException" and "already exists" in str(e):
        # Memory가 이미 존재하면 ID 검색
        memories = memory_client.list_memories()
        memory_id = next((m["id"] for m in memories if m["id"].startswith(memory_name)), None)
        logger.info(f"Memory already exists. Using existing memory ID: {memory_id}")
except Exception as e:
    # Memory 생성 중 발생한 오류 표시
    logger.error(f"❌ ERROR: {e}")
    import traceback

    traceback.print_exc()
    # 오류 발생 시 정리 - 일부 생성된 Memory 삭제
    if "memory_id" in locals() and memory_id:
        try:
            memory_client.delete_memory_and_wait(memory_id=memory_id)
            logger.info(f"Cleaned up memory: {memory_id}")
        except Exception as cleanup_error:
            logger.error(f"Failed to clean up memory: {cleanup_error}")

## 3. Memory 지원 Agent 생성

이 섹션에서는 사용자 지정 Hook으로 Memory가 통합된 Strands Agents framework 기반 Agent를 구축합니다. 이 Agent는 AgentCore Memory에서 메시지를 저장하고 검색하여 대화 컨텍스트를 유지합니다.

> **Memory가 중요한 이유**: AgentCore Runtime의 세션은 일정 시간이 지나면 만료되어 대화 컨텍스트가 삭제됩니다. 대화를 Memory에 저장하면 세션 간에 이전 정보가 유지되므로 오랜 시간이 지난 뒤에도 사용자에게 자연스러운 경험을 제공할 수 있습니다.

### Agent 기능

Agent는 다음 작업을 수행합니다.
1. 각 사용자 및 Assistant 메시지를 Memory에 자동 저장
2. 기존 세션을 이어 갈 때 과거 대화 기록 검색
3. 동일한 사용자와의 여러 상호작용에서 컨텍스트 유지
4. 사용자 Identity 검증을 통해 서로 다른 사용자의 대화 격리

### 구현의 주요 구성 요소

#### 1. Memory Hook Provider
사용자 지정 Hook Provider는 다음을 구현합니다.
- `on_agent_initialized`: Agent 시작 시 trigger되어 AgentCore Memory에서 대화 기록 검색
- `on_message_added`: 대화에 새 메시지가 추가될 때 trigger되어 AgentCore Memory에 저장

#### 2. Agent 초기화
`initialize_agent` 함수는 다음을 수행합니다.
- 올바른 리전으로 Memory Hook 구성
- 적절한 상태 변수(memory_id, actor_id, session_id)로 Agent 설정
- LLM의 system prompt 구성

#### 3. 사용자 검증
`get_user_sub` 함수는 다음을 수행합니다.
- JWKS로 Cognito access token을 검증하고 사용자의 sub(고유 ID)를 반환합니다.

#### 4. Entry Point Handler
runtime_memory_agent 함수는 다음을 수행합니다.
- 입력 payload를 파싱하고 사용자 메시지 추출
- Cognito의 JWT token으로 사용자 Identity 검증
- Agent 초기화 및 세션 추적 관리
- 적절한 컨텍스트로 Agent 호출 처리
- Runtime 환경에 형식이 지정된 응답 반환

Agent 파일을 생성합니다.

In [ ]:
%%writefile runtime_identity_memory_agent.py
import os
import jwt
import json
import logging
from strands import Agent
from jwt import PyJWKClient
from typing import Dict, Any
from strands.models import BedrockModel
from bedrock_agentcore.memory import MemoryClient
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from strands.hooks import AgentInitializedEvent, HookProvider, HookRegistry, MessageAddedEvent

# 상세 로깅 구성
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(name)s: %(message)s'
)
logger = logging.getLogger("runtime-memory-agent")

# AgentCore app 초기화
app = BedrockAgentCoreApp()

MODEL_ID = os.getenv('MODEL_ID')
MEMORY_ID = os.getenv('MEMORY_ID')
COGNITO_USER_POOL = os.getenv('COGNITO_USER_POOL')
REGION = os.getenv('AWS_REGION')

# 전역 Agent 인스턴스 - 첫 요청에서 초기화
agent = None

class MemoryHookProvider(HookProvider):
    """Bedrock Memory와 통합하는 사용자 지정 훅 제공자입니다."""
    
    def __init__(self, region_name):
        logger.info(f"Initializing MemoryHookProvider with region {region_name}")
        self.memory_client = MemoryClient(region_name=region_name)
    
    def on_agent_initialized(self, event: AgentInitializedEvent):
        """에이전트가 시작될 때 최근 대화 기록을 불러옵니다."""
        logger.info("Agent initialization hook triggered")
        
        memory_id = event.agent.state.get("memory_id")
        actor_id = event.agent.state.get("actor_id")
        session_id = event.agent.state.get("session_id")
        
        logger.info(f"State values - memory_id: {memory_id}, actor_id: {actor_id}, session_id: {session_id}")
        
        missing_values = []
        if not memory_id:
            missing_values.append("memory_id")
        if not actor_id:
            missing_values.append("actor_id")
        if not session_id:
            missing_values.append("session_id")
            
        if missing_values:
            logger.warning(f"Missing required values: {', '.join(missing_values)}")
            return
        
        try:
            # 최대 1개의 event를 나열하여 먼저 세션 존재 여부 확인
            logger.info(f"Checking if session {session_id} exists...")
            session_exists = False
            try:
                events = self.memory_client.list_events(
                    memory_id=memory_id,
                    actor_id=actor_id,
                    session_id=session_id,
                    max_results=1
                )
                session_exists = len(events) > 0
                logger.info(f"Session exists: {session_exists} (found {len(events)} events)")
            except Exception as e:
                logger.warning(f"Error checking session existence: {e}")
                # Event가 없다고 가정하고 계속 진행
                session_exists = False
            
            # 세션이 없으면 대화 기록을 불러올 필요 없음
            if not session_exists:
                logger.info(f"No existing conversation found for session {session_id}")
                return
            
            # 세션이 있으므로 대화 기록 불러오기
            logger.info(f"Loading conversation history for existing session {session_id}")
            recent_turns = self.memory_client.get_last_k_turns(
                memory_id=memory_id,
                actor_id=actor_id,
                session_id=session_id,
                k=5
            )
            
            if recent_turns:
                logger.info(f"✅ Loaded {len(recent_turns)} conversation turns from memory")
                context_messages = []
                for turn in recent_turns:
                    for message in turn:
                        role = message['role']
                        content = message['content']['text']
                        context_messages.append(f"{role}: {content}")
                
                context = "\n".join(context_messages)
                event.agent.system_prompt += f"\n\nRecent conversation:\n{context}"
                logger.info("✅ Added conversation context to system prompt")
            else:
                logger.info("No recent turns found for this session")
                
        except Exception as e:
            logger.error(f"❌ Memory load error: {e}", exc_info=True)
    
    def on_message_added(self, event: MessageAddedEvent):
        """메시지를 메모리에 저장합니다."""
        logger.info("Message added hook triggered")
        
        memory_id = event.agent.state.get("memory_id")
        actor_id = event.agent.state.get("actor_id")
        session_id = event.agent.state.get("session_id")
        
        logger.info(f"State values - memory_id: {memory_id}, actor_id: {actor_id}, session_id: {session_id}")
        
        missing_values = []
        if not memory_id:
            missing_values.append("memory_id")
        if not actor_id:
            missing_values.append("actor_id")
        if not session_id:
            missing_values.append("session_id")
            
        if missing_values:
            logger.warning(f"❌ Cannot save message - missing values: {', '.join(missing_values)}")
            return
            
        messages = event.agent.messages
        try:
            last_message = messages[-1]
            message_content = str(last_message.get("content", ""))
            message_role = last_message["role"]
            
            logger.info(f"Saving {message_role} message to memory: {message_content[:30]}...")
            
            self.memory_client.create_event(
                memory_id=memory_id,
                actor_id=actor_id,
                session_id=session_id,
                messages=[(message_content, message_role)]
            )
            logger.info("✅ Message saved to memory successfully")
        except Exception as e:
            logger.error(f"❌ Memory save error: {e}", exc_info=True)
    
    def register_hooks(self, registry: HookRegistry):
        logger.info("Registering memory hooks")
        registry.add_callback(MessageAddedEvent, self.on_message_added)
        registry.add_callback(AgentInitializedEvent, self.on_agent_initialized)

def initialize_agent(actor_id, session_id):
    """처음 사용할 에이전트를 초기화합니다."""
    global agent
    
    logger.info(f"Initializing agent for actor_id={actor_id}, session_id={session_id}")
    
    # 모델 및 Memory Hook 생성
    logger.info(f"Creating model with ID: {MODEL_ID}")
    model = BedrockModel(model_id=MODEL_ID)
    logger.info(f"Creating memory hook with region: {REGION}")
    memory_hook = MemoryHookProvider(region_name=REGION)
    
    # 적절한 초기 상태로 Agent 생성
    logger.info("Creating agent with memory hook")
    agent = Agent(
        model=model,
        hooks=[memory_hook],
        system_prompt="You're a helpful, memory-enabled agent deployed on AgentCore Runtime. You can remember previous interactions within the same session. Be friendly and concise in your responses.",
        state={
            "memory_id": MEMORY_ID,
            "actor_id": actor_id,
            "session_id": session_id
        }
    )
    logger.info(f"✅ Agent initialized with state: {agent.state.get()}")

def get_user_sub(access_token: str, region: str, user_pool_id: str) -> str:
    """
    JWKS로 Cognito 액세스 토큰을 검증하고 사용자의 sub(고유 ID)를 반환합니다.

    :param access_token: JWT 액세스 토큰 문자열
    :param region: Cognito User Pool의 AWS 리전
    :param user_pool_id: Cognito User Pool ID
    :return: 토큰이 유효하면 사용자의 'sub' 클레임
    :raises jwt.InvalidTokenError: 검증에 실패한 경우
    """
    access_token = access_token[7:]
    jwks_url = f"https://cognito-idp.{region}.amazonaws.com/{user_pool_id}/.well-known/jwks.json"
    jwks_client = PyJWKClient(jwks_url)
    signing_key = jwks_client.get_signing_key_from_jwt(access_token)

    decoded = jwt.decode(
        access_token,
        signing_key.key,
        algorithms=["RS256"],
        issuer=f"https://cognito-idp.{region}.amazonaws.com/{user_pool_id}",
        options={"require": ["exp", "iat", "iss", "token_use"]}
    )

    if decoded.get("token_use") != "access":
        raise jwt.InvalidTokenError("Token is not an access token")

    return decoded["sub"]

@app.entrypoint
def runtime_memory_agent(payload, context):
    """
    메모리 지원 에이전트의 기본 진입점입니다.
    
    인자:
        payload: 사용자 데이터가 포함된 입력 페이로드
        context: 세션 정보가 포함된 Runtime 컨텍스트 객체
    """
    global agent
    
    # payload와 context 정보를 모두 기록
    logger.info(f"Received payload: {payload}")
    logger.info(f"Context: {context}")
    logger.info(f"Context Auth: {context.request_headers.get('Authorization')}")
    logger.info(f"User Sub: {get_user_sub(context.request_headers.get('Authorization'), REGION, COGNITO_USER_POOL)}")
    logger.info(f"Context session_id: {context.session_id}")
    
    # 필수 값 추출 및 검증
    user_input = payload.get("prompt")
    actor_id = get_user_sub(context.request_headers.get('Authorization'), REGION, COGNITO_USER_POOL)
    session_id = context.session_id  # context에서 session_id 가져오기
    
    # 필수 field 검증
    if user_input is None:
        error_msg = "❌ ERROR: Missing 'prompt' field in payload"
        logger.error(error_msg)
        return error_msg
    
    # 첫 요청에서 Agent 초기화
    if agent is None:
        logger.info("First request - initializing agent")
        initialize_agent(actor_id, session_id)
    else:
        logger.info("Using existing agent instance")
        # Session ID가 변경된 경우 업데이트
        if agent.state.get("session_id") != session_id:
            logger.info(f"Updating session ID to {session_id}")
            agent.state.set("session_id", session_id)
        if agent.state.get("actor_id") != actor_id:
            logger.info(f"Updating actor ID to {actor_id}")
            agent.state.set("actor_id", actor_id)
    
    # 사용자 입력으로 Agent 호출
    logger.info(f"Invoking agent with input: {user_input}")
    response = agent(user_input)
    response_text = response.message['content'][0]['text']
    logger.info(f"✅ Agent response: {response_text[:50]}...")
    
    return response_text

if __name__ == "__main__":
    logger.info("Starting AgentCore application")
    app.run()

## 4. AgentCore Runtime에 배포

이 섹션에서는 확장성과 간소화된 운영을 제공하는 관리형 Agent runtime 환경인 Amazon Bedrock AgentCore Runtime에 Agent를 배포합니다. AgentCore Runtime이 복잡한 인프라를 처리하므로 배포가 아니라 Agent logic에 집중할 수 있습니다.

수동 서버 설정과 관리가 필요한 기존 배포 방식과 달리 AgentCore Runtime은 코드를 container로 자동 packaging하여 AWS 인프라에 배포하고 호출용 보안 HTTPS endpoint를 제공합니다. 이 접근 방식은 Agent가 수요에 맞춰 확장되고 프로덕션 환경에서 안정적으로 작동하도록 보장합니다.

### 내부 동작

AgentCore Runtime에 배포하면 다음 작업이 자동으로 수행됩니다.
1. 코드를 Docker container image로 packaging
2. Container image를 Amazon ECR(Elastic Container Registry)에 push
3. Agent 실행용 AWS Lambda 함수 또는 container 서비스 provision
4. 보안 액세스를 위한 API Gateway endpoint 생성
5. 안전한 운영을 위한 IAM 역할 및 권한 구성

### 알아야 할 사항

- **AgentCore Runtime**은 Agent를 Docker container로 packaging하여 관리형 AWS 인프라에 배포합니다.
- **환경 변수**는 Agent를 구성합니다.
- `MEMORY_ID`: 앞서 생성한 Memory 리소스
- `MODEL_ID`: Claude 3.5 Haiku 모델 ID
- `AWS_REGION`: 배포할 AWS 리전
- `COGNITO_USER_POOL`: 인증용 Cognito User Pool

> 💡 **팁**: AgentCore starter toolkit은 IAM 역할, ECR repository, container build를 포함한 복잡한 배포 단계를 모두 처리합니다.

### 배포 구성

배포 구성을 설정합니다.

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime

agentcore_runtime = Runtime()
agent_name = f"runtime_memory_agent_{unique_id}"

response = agentcore_runtime.configure(
    entrypoint="runtime_identity_memory_agent.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=REGION,
    agent_name=agent_name,
    non_interactive=True,
    memory_mode="NO_MEMORY",
    request_header_configuration={"requestHeaderAllowlist": ["Authorization"]},
    authorizer_configuration={
        "customJWTAuthorizer": {
            "discoveryUrl": cognito_config.get("discovery_url"),
            "allowedClients": [cognito_config.get("client_id")],
        }
    },
)
response

### Agent 실행

이제 Agent를 AgentCore Runtime에 실행합니다. 이 단계에서는 구성된 Agent를 AgentCore의 관리형 인프라에 배포합니다. 이 과정에서 앞서 생성한 Memory ID, 사용할 모델 ID, AWS 리전, 인증용 Cognito User Pool ID 등 Agent에 필요한 필수 환경 변수도 전달합니다.

배포 후에는 사용자 메시지로 호출할 수 있는 보안 endpoint를 통해 Agent에 액세스할 수 있습니다. Endpoint는 Cognito 인증으로 보호되므로 권한이 있는 사용자만 Agent에 액세스할 수 있습니다.

In [ ]:
launch_result = agentcore_runtime.launch(
    env_vars={
        "MEMORY_ID": memory_id,
        "MODEL_ID": "global.anthropic.claude-haiku-4-5-20251001-v1:0",
        "AWS_REGION": REGION,
        "COGNITO_USER_POOL": cognito_config["pool_id"],
    }
)

### 배포 상태 확인

Agent의 배포 상태를 확인합니다. AgentCore Runtime이 container를 build하고 필요한 리소스를 provision한 뒤 Agent를 AWS 인프라에 배포하므로 몇 분 정도 걸릴 수 있습니다. 배포가 완료될 때까지 10초마다 상태를 polling합니다.

In [ ]:
status_response = agentcore_runtime.status()
status = status_response.endpoint["status"]
end_status = ["READY", "CREATE_FAILED", "DELETE_FAILED", "UPDATE_FAILED"]

while status not in end_status:
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint["status"]
    print(f"Current status: {status}")

if status == "READY":
    print("✅ Agent successfully deployed!")
else:
    print(f"❌ Deployment ended with status: {status}")

## 5. Agent 테스트

Agent가 배포되었으므로 메시지를 전송하여 이전 상호작용을 기억하는지 테스트합니다. 또한 한 사용자의 대화가 다른 사용자에게 보이지 않도록 서로 다른 사용자의 Memory 컨텍스트가 격리되는지 확인합니다.

**세션 관리에 관한 중요 참고 사항**

- **세션 관리**: Session ID를 제공하지 않으면 AgentCore Runtime이 자동으로 생성하지만, 애플리케이션에서 Session ID를 명시적으로 관리하는 것이 좋습니다. 이를 통해 다음 항목을 더 효과적으로 제어할 수 있습니다.
- 세션 timeout 후 대화 계속
- 적절한 시점에 새 세션 생성(예: 사용자가 새 대화를 시작)
- 동일한 사용자의 여러 병렬 대화 처리
- 애플리케이션 요구 사항에 따른 세션 만료 policy 구현

- **메모리 지속성**: AgentCore Runtime에서 세션이 만료되더라도 동일한 사용자가 새 세션을 시작하면 Agent가 AgentCore Memory에서 이전 대화를 검색할 수 있습니다.

먼저 JWT token을 검증할 helper 함수를 정의합니다.

In [ ]:
def test_user_memory_isolation():
    """
    AgentCore에서 사용자별 메모리가 격리되는지 테스트합니다.

    이 테스트에서는 다음을 검증합니다.
    1. 각 사용자의 대화가 별도로 저장됨
    2. 에이전트가 각 사용자와의 이전 상호 작용을 기억함
    3. 서로 다른 사용자 간에 사용자 데이터가 공유되지 않음
    """
    print("\n" + "=" * 50)
    print("USER MEMORY ISOLATION TEST")
    print("=" * 50)

    # testuser1 및 testuser2용 Session ID 생성
    testuser1_session_id = f"agent-session-testuser1-{int(time.time())}"
    testuser2_session_id = f"agent-session-testuser2-{int(time.time())}"

    testuser1_token = cognito_config["bearer_tokens"]["testuser1"]
    testuser2_token = cognito_config["bearer_tokens"]["testuser2"]

    # 1단계: testuser1이 선호하는 색상 공유
    print("\n" + "-" * 50)
    print("STEP 1: First user shares personal information")
    print("-" * 50)
    print('testuser1: "My favorite color is purple."')

    response1 = agentcore_runtime.invoke(
        {"prompt": "My favorite color is purple."},
        session_id=testuser1_session_id,
        bearer_token=testuser1_token,
    )
    print(f'Agent: "{response1["response"]}"')

    # 2단계: testuser2가 선호하는 음식 공유
    print("\n" + "-" * 50)
    print("STEP 2: Second user shares different information")
    print("-" * 50)
    print('testuser2: "My favorite food is pizza."')

    response2 = agentcore_runtime.invoke(
        {"prompt": "My favorite food is pizza."},
        session_id=testuser2_session_id,
        bearer_token=testuser2_token,
    )
    print(f'Agent: "{response2["response"]}"')

    # 3단계: testuser1이 선호하는 색상 질문
    print("\n" + "-" * 50)
    print("STEP 3: First user tests agent's memory")
    print("-" * 50)
    print('testuser1: "What did I say my favorite color was?"')

    response3 = agentcore_runtime.invoke(
        {"prompt": "What did I say my favorite color was?"},
        session_id=testuser1_session_id,
        bearer_token=testuser1_token,
    )
    print(f'Agent: "{response3["response"]}"')

    # 4단계: testuser2가 선호하는 음식 질문
    print("\n" + "-" * 50)
    print("STEP 4: Second user tests agent's memory")
    print("-" * 50)
    print('testuser2: "What\'s my favorite food?"')

    response4 = agentcore_runtime.invoke(
        {"prompt": "What's my favorite food?"},
        session_id=testuser2_session_id,
        bearer_token=testuser2_token,
    )
    print(f'Agent: "{response4["response"]}"')

    # 5단계: testuser1이 음식에 관해 질문(알지 못해야 함)
    print("\n" + "-" * 50)
    print("STEP 5: Testing memory isolation (first user)")
    print("-" * 50)
    print('testuser1: "What\'s my favorite food?"')

    response5 = agentcore_runtime.invoke(
        {"prompt": "What's my favorite food?"},
        session_id=testuser1_session_id,
        bearer_token=testuser1_token,
    )
    print(f'Agent: "{response5["response"]}"')

    # 6단계: testuser2가 색상에 관해 질문(알지 못해야 함)
    print("\n" + "-" * 50)
    print("STEP 6: Testing memory isolation (second user)")
    print("-" * 50)
    print('testuser2: "What\'s my favorite color?"')

    response6 = agentcore_runtime.invoke(
        {"prompt": "What's my favorite color?"},
        session_id=testuser2_session_id,
        bearer_token=testuser2_token,
    )
    print(f'Agent: "{response6["response"]}"')

In [ ]:
test_user_memory_isolation()

## 핵심 개념

이 튜토리얼에서는 AgentCore로 Memory 지원 Agent를 구축하는 데 필요한 몇 가지 중요 개념을 학습했습니다.

1. **Memory 통합**: Amazon Bedrock Memory를 사용하여 세션 전반의 대화 기록을 저장하고, 세션이 만료되어도 Agent가 시간 경과에 따라 컨텍스트를 유지하도록 하는 방법

2. **세션 관리**: Session ID를 사용하여 대화를 구성하고 사용자가 돌아왔을 때 관련 기록을 검색하여 자연스러운 경험을 제공하는 방법

3. **AgentCore 배포**: 확장, 보안, 인프라 관리를 자동으로 처리하는 프로덕션 Runtime 환경에 Agent를 배포하는 방법

4. **Memory Hook**: Memory 서비스와 통합되는 사용자 지정 Hook을 구현하여 Agent 수명 주기의 특정 시점에 대화 기록을 저장하고 검색하는 방법

5. **사용자 Identity 및 개인정보 보호**: 인증을 사용하여 각 사용자의 대화 기록이 비공개로 유지되고 다른 사용자와 격리되도록 하는 방법

이러한 개념은 지속형 Memory와 정교한 대화 관리 기능을 갖춘 더 복잡한 Agent를 구축하는 기반이 됩니다.

## 리소스 정리(선택 사항)

이 튜토리얼에서 생성한 리소스가 더 이상 필요하지 않으면 불필요한 AWS 요금을 방지하기 위해 정리할 수 있습니다. 정리 대상은 다음과 같습니다.

1. AgentCore Runtime Agent
2. Agent container image를 포함한 ECR repository
3. 대화 기록을 저장하는 Memory 리소스

먼저 리소스를 확인합니다.

In [ ]:
# 리소스 식별자 가져오기
if "launch_result" in locals():
    print(f"Agent ID: {launch_result.agent_id}")
    print(f"ECR Repository: {launch_result.ecr_uri.split('/')[1]}")
else:
    print("Launch results not available")

In [ ]:
# 모든 리소스를 삭제하려는 경우에만 이 셀 실행

# 1. AgentCore Runtime 삭제
if "launch_result" in locals() and hasattr(launch_result, "agent_id"):
    try:
        agentcore_control_client = boto3.client("bedrock-agentcore-control", region_name=REGION)

        runtime_delete_response = agentcore_control_client.delete_agent_runtime(
            agentRuntimeId=launch_result.agent_id,
        )
        print(f"✅ Deleted AgentCore Runtime: {launch_result.agent_id}")
    except Exception as e:
        print(f"❌ Error deleting AgentCore Runtime: {e}")
else:
    print("No AgentCore Runtime to delete")

# 2. ECR repository 삭제
if "launch_result" in locals() and hasattr(launch_result, "ecr_uri"):
    try:
        ecr_client = boto3.client("ecr", region_name=REGION)

        repository_name = launch_result.ecr_uri.split("/")[1]
        response = ecr_client.delete_repository(
            repositoryName=repository_name,
            force=True,  # image가 있어도 강제 삭제
        )
        print(f"✅ Deleted ECR repository: {repository_name}")
    except Exception as e:
        print(f"❌ Error deleting ECR repository: {e}")
else:
    print("No ECR repository to delete")

# 3. Memory resource 삭제
if "memory_id" in locals() and memory_id:
    try:
        memory_client = MemoryClient(region_name=REGION)
        memory_client.delete_memory_and_wait(memory_id=memory_id)
        print(f"✅ Deleted memory resource: {memory_id}")
    except Exception as e:
        print(f"❌ Error deleting memory resource: {e}")
else:
    print("No memory resource to delete")

# 4. Cognito User Pool 및 관련 resource 삭제
if "cognito_config" in locals() and cognito_config and "pool_id" in cognito_config:
    try:
        cognito_client = boto3.client("cognito-idp", region_name=REGION)

        # User Pool ID 가져오기
        pool_id = cognito_config["pool_id"]

        # 모든 User Pool Client 나열 및 삭제
        clients_response = cognito_client.list_user_pool_clients(UserPoolId=pool_id, MaxResults=60)

        for client in clients_response.get("UserPoolClients", []):
            client_id = client["ClientId"]
            cognito_client.delete_user_pool_client(UserPoolId=pool_id, ClientId=client_id)
            print(f"✅ Deleted User Pool Client: {client_id}")

        # User Pool 자체 삭제
        cognito_client.delete_user_pool(UserPoolId=pool_id)
        print(f"✅ Deleted Cognito User Pool: {pool_id}")

    except Exception as e:
        print(f"❌ Error deleting Cognito resources: {e}")
else:
    print("No Cognito resources to delete")

print("\n✅ Cleanup complete")

## 축하합니다!

Amazon Bedrock AgentCore Runtime, AgentCore Identity, AgentCore Memory를 사용하여 첫 번째 Memory 지원 Agent를 성공적으로 구축하고 배포했습니다. 이 Agent는 몇 가지 중요한 기능을 보여 줍니다.

1. **메모리 지속성**: Agent가 이전 대화를 기억할 수 있습니다.
2. **사용자 Identity**: Agent가 사용자별로 분리된 대화 기록을 유지합니다.
3. **관리형 인프라**: Agent가 AWS 관리형 인프라에서 실행되며 필요에 따라 자동으로 확장됩니다.

### 다음 단계

기본 사항을 이해했으므로 다음과 같은 방법으로 Agent를 개선할 수 있습니다.

1. **도구 추가**: 계산기, database connector 또는 API 호출 같은 도구로 Agent가 대화 외의 작업도 수행하도록 개선
2. **Memory 개선**: 장기 메모리를 사용하는 더 정교한 Memory strategy 구현
3. **UI 구축**: React, Flutter 또는 Swift 같은 framework로 Agent용 web 또는 mobile interface 생성
4. **비즈니스 logic 추가**: Agent를 CRM, knowledge base 또는 내부 도구 같은 비즈니스 시스템과 통합